In [ ]:
import os
import argparse
import pandas as pd
from tqdm import tqdm
import os, re, json
from typing import List, Dict, Any
import time

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
from vllm import LLM, SamplingParams

In [ ]:
MODEL_DIR = "/shares/animalwelfare.crs.uzh/llms/DeepSeek-R1-Distill-Qwen-32B"  # contains config.json, tokenizer.*, *.safetensors

# 2) Keep vLLM/HF offline at runtime
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

In [ ]:
llm = LLM(
    model=MODEL_DIR,
    tokenizer=MODEL_DIR,
    trust_remote_code=True,
    #dtype="bfloat16",         # or "float16"
    max_model_len=8192,
    tensor_parallel_size=1,
)


In [ ]:
def build_prompt(abstract, SYSTEM, PROMPT_GUIDE, SCHEMA, FEW_SHOT_EXAMPLES):
    """
    Build a strict instruction prompt specialized for preclinical animal studies.
    - Preserves your schema and <json> wrapper.
    - Enforces ordering, no entity changes, ≤12-word evidence quotes.
    """

    return (
        f"SYSTEM:\n{SYSTEM}\n\n"
        "USER:\n"
        f"{PROMPT_GUIDE}\n\n"
        "### OUTPUT FORMAT ###\n"
        "- RETURN ONLY A JSON OBJECT INSIDE <json> AND </json> TAGS, MATCHING THIS SCHEMA:\n"
        f"{SCHEMA}\n\n"
        f"{FEW_SHOT_EXAMPLES}\n\n"
        f"NEW Indication:\n<<<{abstract}>>>\n\n"
        "Return your answer strictly as JSON inside <json>...</json>, nothing else:\n<<<\n"
        
    )

In [ ]:
SYSTEM = (
    "YOU ARE A WORLD-CLASS MEDICAL NAMED ENTITY RECOGNITION (NER) EXPERT. YOUR ROLE IS TO EXTRACT **DISEASE NAMES AND CLINICALLY RELEVANT CONDITIONS** FROM DRUG INDICATION TEXTS WITH EXTREME PRECISION."
)

SCHEMA_DISEASE = """
<json>
{
  "entities": [
    {
      "text": "DISEASE_OR_CONDITION_1"
    },
    {
      "text": "DISEASE_OR_CONDITION_2"
    }
  ]
}
</json>
""".strip()

GUIDE_DISEASE = r"""

FOLLOW THIS STEP-BY-STEP REASONING PIPELINE FOR EACH EXTRACTION:

1. **UNDERSTAND**: READ the full "INDICATIONS AND USAGE" text carefully.
2. **BASICS**: IDENTIFY medically treatable targets such as:
   - Diseases (e.g., “diabetes mellitus”)
   - Disorders (e.g., “anxiety disorder”)
   - Syndromes (e.g., “post-herpetic neuralgia”)
   - Clinical symptoms IF they are **explicit therapeutic targets** (e.g., “pain”)
3. **BREAK DOWN**: PARSE the sentence into components that describe WHAT is being treated or relieved.
4. **ANALYZE**:
   - INCLUDE all forms of disease mentions: general, specific, or abbreviated (e.g., “PHN”)
   - EXPAND abbreviations to full forms when present
5. **BUILD**: CREATE a JSON object with a list of extracted entities, EACH as a separate object with `"text"` key.
6. **EDGE CASES**:
   - INCLUDE general symptoms (e.g., “pain”, “inflammation”) **only if** explicitly listed as indications
   - DO NOT remove duplicate medical concepts with distinct phrasing (e.g., "anxiety disorder" and "anxiety")
7. **FINAL ANSWER**: OUTPUT CLEAN JSON. DO NOT INCLUDE NON-MEDICAL WORDS, DRUG NAMES, OR MECHANISMS OF ACTION.

- DO NOT return an empty entity list — use `"Need more information"`
- DO NOT hallucinate conditions that are not explicitly or clearly implied
"""

FEW_SHOT_EXAMPLES_DISEASE = r"""
### FEW-SHOT EXAMPLES (ADAPTED TO SCHEMA) ###

####  EXAMPLE 1  
**INPUT**  
```
INDICATIONS Diazepam is indicated for the management of anxiety disorders or for the short-term relief of the symptoms of anxiety.
```

**OUTPUT**

<json>
{
  "entities": [
    {
      "text": "anxiety disorders"
    },
    {
      "text": "anxiety"
    }
  ]
}
</json>


---

#### EXAMPLE 2  
**INPUT**  
```
INDICATIONS AND USAGE: Captopril tablets are indicated for the treatment of hypertension.
```

**OUTPUT**

<json>
{
  "entities": [
    {
      "text": "hypertension"
    }
  ]
}

```

---

####  EXAMPLE 3  
**INPUT**  
```
INDICATIONS AND USAGE: ZTLIDO is indicated for relief of pain associated with post-herpetic neuralgia (PHN) in adults.
```

**OUTPUT**

<json>
{
  "entities": [
    {
      "text": "pain"
    },
    {
      "text": "post-herpetic neuralgia"
    }
  ]
}
</json>

####  EXAMPLE 4 

**INPUT**  
```
Indications and Usage Intravenous adenosine injection is indicated for the following
```

**OUTPUT**

<json>
{
  "entities": ["Need more information"]
}
</json>

---
""".strip()

In [ ]:
def extract_json(text: str):
    # 1. Preferred case: inside <json>...</json>
    m = re.search(r"<json>\s*(\{.*\})\s*(?:</json>)?", text, flags=re.S)
    if m:
        #print(m.group(1))
        return json.loads(m.group(1))

    # 2. Fenced code block: ```json ... ```
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S | re.I)
    if m:
        return json.loads(m.group(1))

    # 3. Any last JSON-looking object in the string
    return _parse_last_json_anywhere(text)


def _parse_last_json_anywhere(text: str):
    """
    Try to grab the last {...} block from the text and parse it as JSON.
    Useful if LLM prepends explanations or stray text.
    """
    matches = list(re.finditer(r"\{.*\}", text, flags=re.S))
    if not matches:
        raise ValueError(f"No JSON object found in: {text}")
    for m in reversed(matches):  # last one usually the valid one
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            continue
    raise ValueError(f"Could not parse JSON from: {text}")

In [ ]:
sampling = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=2000,
    stop=["</json>"],   
)

In [ ]:
indication = (
"To treat ALK-positive lung cancer Drug Trials Snapshot")
# Build a disease-specific prompt
prompt = build_prompt(
    indication,
    SYSTEM,
    GUIDE_DISEASE,
    SCHEMA_DISEASE,
    FEW_SHOT_EXAMPLES_DISEASE
)



out = llm.generate([prompt], sampling)
text = out[0].outputs[0].text  # already stops before </json>
data = extract_json(text)
print(json.dumps(data, indent=2))

In [ ]:
found_entities = [
                ent["text"] for ent in data.get("entities", [])
                #if ent.get("role") == "primary_target"
            ]
found_entities

In [ ]:
for entity in found_entities:
    if entity in indication:
        print("True")
    else:
        print("False")
               

In [ ]:
def check_partial(entity_parts, indication_text):
    """
    Returns True if at least one word from entity_parts is found in indication_text.
    """
    for part in entity_parts:
        if part.lower() in indication_text:
            return True
    return False
    
def apply_llm_cleanup(row, max_retries=3, with_print=False):
    indication_sent = row['indications_first_sent']
    indication_full = row['indications_and_usage']

    indication = indication_sent


    # Try extraction with retries
    for attempt in range(1, max_retries + 1):
        if attempt > 1:
            indication = indication_full

        prompt = build_prompt(
            indication,
            SYSTEM,
            GUIDE_DISEASE,
            SCHEMA_DISEASE,
            FEW_SHOT_EXAMPLES_DISEASE
        )
        try:
            out = llm.generate([prompt], sampling)
            text = out[0].outputs[0].text
            if with_print:
                print(text)
            data = extract_json(text)  # may raise

            found_entities = [
                ent["text"] for ent in data.get("entities", [])
            ]

            all_entities_in_text = True
            indication_lower = indication.lower()

            for entity in found_entities:
                if entity == "Need more information":
                    all_entities_in_text = False
                    indication = indication_full
                elif entity.lower() not in indication_lower:
                    entity_parts = entity.split(" ")
                    if len(entity_parts) > 1:
                        if check_partial(entity_parts, indication_lower):
                            continue
                        else:
                            all_entities_in_text = False
                            indication = indication_full

            # Validation: all extracted must be in the original set
            if all_entities_in_text:
                result = "|".join(found_entities)
                print(f"{row["application_number"]} output: {result} from {indication_sent}")
                return result, indication
            else:
                print(f"⚠️ attempt {attempt}: extracted entities not in original text or more information needed → {found_entities} from {indication}")
                if attempt == max_retries:
                    print("❌ Giving up after max retries. Returning original entities.")
                    return "", indication
                continue

        except Exception as e:
            print(f"❌ attempt {attempt} failed: {e}")
            indication = indication_full
            if attempt == max_retries:
                print("⚠️ Giving up after max retries. Returning original entities.")
                return "", indication

### Full FDA dataset

In [ ]:
drugs_with_indications = pd.read_csv("out/drugs_with_indications.csv")
print(drugs_with_indications.shape)

# Randomly sample 50 rows
sample_df = drugs_with_indications.sample(n=50, random_state=42)

# (Optional) reset index for convenience
sample_df = sample_df.reset_index(drop=True)

# (Optional) save to a new file
sample_df.to_csv("out/drugs_with_indications_sample50.csv", index=False)

In [ ]:
dups = drugs_with_indications[
    drugs_with_indications.duplicated(subset=["unique_id","indications_first_sent"], keep=False)
]
dups.head()

In [ ]:
drugs_with_indications.head()

In [ ]:
import pandas as pd
from tqdm import tqdm

new_col_name = "disease_from_indications"
prior_path   = "out/drugs_with_indications_LLM_cleaned_20260113.csv"
save_path    = "out/drugs_with_indications_LLM_cleaned_20260131.csv"
batch_save_every = 500

# ---------------------------
# Load prior results
# ---------------------------
try:
    prior_df = pd.read_csv(prior_path, dtype=str)
    print("prior:", prior_df.shape)
except FileNotFoundError:
    prior_df = pd.DataFrame(columns=["unique_id", new_col_name])
    print(f"ℹ️ Prior file not found: {prior_path}. Starting fresh.")

# Make sure unique_id exists in both
if "unique_id" not in drugs_with_indications.columns:
    raise ValueError("drugs_with_indications must contain a 'unique_id' column.")

if new_col_name not in drugs_with_indications.columns:
    drugs_with_indications[new_col_name] = ""

# ---------------------------
# Split into already-done vs to-process (by unique_id)
# ---------------------------
processed_ids = set(prior_df["unique_id"].dropna().astype(str))

mask_done = drugs_with_indications["unique_id"].astype(str).isin(processed_ids)
df_done = drugs_with_indications.loc[mask_done].copy()
df_todo = drugs_with_indications.loc[~mask_done].copy()

print(f"done (in prior): {df_done.shape} | todo: {df_todo.shape}")

# Fill df_done from prior (so it has the column populated)
if not prior_df.empty:
    prior_map = (
        prior_df.dropna(subset=["unique_id"])
                .drop_duplicates(subset=["unique_id"], keep="last")
                .set_index("unique_id")[new_col_name]
                .fillna("")
                .to_dict()
    )
    df_done[new_col_name] = df_done["unique_id"].astype(str).map(prior_map).fillna(df_done[new_col_name].fillna(""))


In [ ]:
12858 + 7043

In [ ]:
# ---------------------------
# Helpers
# ---------------------------
def _norm_text(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return " ".join(str(s).strip().split()).lower()

# Ensure required text cols exist in df_todo
for c in ["indications_first_sent", "indications_and_usage"]:
    if c not in df_todo.columns:
        df_todo[c] = ""

# Precompute keys on df_todo only
sent_key_series = df_todo["indications_first_sent"].map(_norm_text)
full_key_series = df_todo["indications_and_usage"].map(_norm_text)

# Output buffer for df_todo
out = df_todo[new_col_name].astype(str).fillna("").tolist()

# semantic cache: normalized text -> (result, used_text)
semantic_cache_by_text = {}

processed_since_save = 0
cache_hits = 0
llm_calls = 0
errors = 0

# ---------------------------
# Process only TODO rows
# ---------------------------
for idx, row in tqdm(
    enumerate(df_todo.itertuples(index=False)),
    total=len(df_todo)
):
    # Skip if already filled (e.g., from earlier work)
    if out[idx].strip():
        continue

    sent_key = sent_key_series.iat[idx]
    full_key = full_key_series.iat[idx]

    try:
        hit = None
        if sent_key and sent_key in semantic_cache_by_text:
            hit = semantic_cache_by_text[sent_key]
        elif full_key and full_key in semantic_cache_by_text:
            hit = semantic_cache_by_text[full_key]

        if hit is not None:
            result, used_text = hit
            cache_hits += 1
        else:
            row_dict = row._asdict()
            result, used_text = apply_llm_cleanup(row_dict)
            llm_calls += 1

            used_key = _norm_text(used_text)
            semantic_cache_by_text[used_key] = (result, used_text)

            # map also sent/full keys to help reuse later
            if sent_key and sent_key != used_key:
                semantic_cache_by_text[sent_key] = (result, used_text)
            if full_key and full_key not in (used_key, sent_key):
                semantic_cache_by_text[full_key] = (result, used_text)

        out[idx] = result or ""

    except Exception as e:
        errors += 1
        out[idx] = out[idx] or ""
        if errors <= 5:
            uid = getattr(row, "unique_id", None)
            print(f"⚠️ Error at todo-row {idx} (unique_id={uid}): {e}")

    processed_since_save += 1
    if processed_since_save >= batch_save_every:
        df_todo[new_col_name] = out

        # concat partial results and save (so you can resume easily)
        tmp_all = pd.concat([df_done, df_todo], ignore_index=True)
        tmp_all = tmp_all.drop_duplicates(subset=["unique_id"], keep="last")

        tmp_all.to_csv(save_path, index=False)
        print(f"✅ Saved progress | cache_hits={cache_hits} llm_calls={llm_calls} errors={errors}")

        processed_since_save = 0

# finalize df_todo
df_todo[new_col_name] = out

# ---------------------------
# Concat all + dedupe + save
# ---------------------------
all_df = pd.concat([df_done, df_todo], ignore_index=True)
all_df = all_df.drop_duplicates(subset=["unique_id"], keep="last")

all_df.to_csv(save_path, index=False)

print(f"🎉 Done | total={all_df.shape} todo={df_todo.shape} cache_hits={cache_hits} llm_calls={llm_calls} errors={errors}")